In [1]:
import qubx
from qubx import logger

%qubxd

%load_ext autoreload
%autoreload 2

from qubx.backtester.simulator import simulate
from qubx.core.basics import DataType, Signal
from qubx.core.interfaces import IStrategy, IStrategyContext, IStrategyInitializer, MarketEvent, TriggerEvent
from qubx.core.lookups import lookup
from qubx.data.registry import StorageRegistry
from qubx.data.transformers import TypedGenericSeries


⠀⠀⡰⡖⠒⠒⢒⢦⠀⠀   
⠀⢠⠃⠈⢆⣀⣎⣀⣱⡀  QUBX | Quantitative Backtesting Environment 
⠀⢳⠒⠒⡞⠚⡄⠀⡰⠁         (c) 2025, ver. 0.7.40.dev8
⠀⠀⠱⣜⣀⣀⣈⣦⠃⠀⠀⠀ 
        


## Datareaders 

In [3]:
stor = StorageRegistry.get("csv::../../tests/data/storages/csv/")
stor.get_exchanges()

['BINANCE.UM', 'HYPERLIQUID', 'KRAKEN']

In [ ]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("AAVEUSDT", "ohlc(1h)")

In [ ]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("BTCUSDT", "quote")

In [ ]:
stor.get_reader("KRAKEN", "SWAP").get_time_range("AAVEUSD", "ohlc(1h)")

In [ ]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote").to_records()[:3]

In [ ]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote", None, None).transform(TypedGenericSeries())

In [11]:
_X1 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 1000, "high": 1100, "low": 950, "close": 1050 },
    "2025-01-01 01:00": {"open": 1050, "high": 1200, "low": 900, "close": 1075 },
    "2025-01-01 02:00": {"open": 1075, "high": 1300, "low": 1050, "close": 1100 },
}, orient="index")
_X1.index = pd.DatetimeIndex(_X1.index)

_X2 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 100, "high": 110, "low": 95, "close": 105 },
    "2025-01-01 01:00": {"open": 105, "high": 120, "low": 90, "close": 107 },
    "2025-01-01 02:00": {"open": 107, "high": 130, "low": 105, "close": 110 },
}, orient="index")
_X2.index = pd.DatetimeIndex(_X2.index)

In [ ]:
stor = StorageRegistry.get("handy", data={
    "BINANCE.UM:SWAP:BTCUSDT": _X1,
    "BINANCE.UM:SWAP:ETHUSDT": _X2
})
r = stor["BINANCE.UM", "SWAP"]
r.read(["BTCUSDT", "ETHUSDT"], "ohlc(1h)")

## Smoke test run

In [2]:
stor = StorageRegistry.get("csv::../../tests/data/storages/multi/")
# stor = StorageRegistry.get("qdb::quantlab")
stor.get_exchanges()

['BINANCE.UM', 'KRAKEN.F', 'HYPERLIQUID']

In [3]:
rr = stor.get_reader("BINANCE.UM", "SWAP")
print(rr.get_data_types("BTCUSDT"))
rr.read("BTCUSDT", "features", "2026-01-01", "now").to_records()[:4]

[quote, 'ohlc(1h)', 'features']


[[2026-01-01T01:00:00.000000000]	 {'f1': 1.0, 'f2': 11.0},
 [2026-01-01T02:00:00.000000000]	 {'f1': 2.0, 'f2': 21.0},
 [2026-01-01T03:00:00.000000000]	 {'f1': 3.0, 'f2': 31.0},
 [2026-01-01T04:00:00.000000000]	 {'f1': 4.0, 'f2': 41.0}]

In [4]:
class Test1(IStrategy):
    def on_init(self, initializer: IStrategyInitializer):
        initializer.set_event_schedule("1h -1s")

        initializer.set_base_subscription("ohlc(1h)")
        # initializer.set_base_subscription("ohlc_quotes(1h)")
        # initializer.subscribe("ohlc_trades(1h)")

        # initializer.set_base_subscription("ohlc_trades(1h)")
        # initializer.subscribe("ohlc_quotes(1h)")

        # initializer.set_base_subscription("ohlc_trades(1h)")
        # initializer.subscribe("features")
        # self._show = True

    def on_market_data(self, ctx: IStrategyContext, data: MarketEvent) -> list[Signal] | Signal | None:
        # qq = ''
        # for i in ctx.instruments:
        #     qq += f" | {i}: {ctx.quote(i).mid_price()}"
        # if data.type == "quote":

        # if self._show and data.type == "quote":
        #     self._show = False
        #     logger.info(data)

        # if data.type == "features":
            # logger.info(data)
        #     logger.info(data)
        #     self._show = True
        # logger.info(data)
        pass

    def on_event(self, ctx: IStrategyContext, event: TriggerEvent) -> list[Signal] | Signal | None:
        # logger.info(" | ".join([f"{i}: {ctx.quote(i).mid_price()}" for i in ctx.instruments]))
        d = ctx.ohlc(ctx.instruments[0], "1h", 10)
        logger.info(f" === Time: <r>{ctx.time()}</r> ===\n<g>" + str(d.pd()[["open","high","low","close","volume"]]) + "</g>\n - - - - - - -")


In [5]:
simulate(
    Test1(), 
    data=stor, capital=1000, 
    # start="2023-06-01", stop="2023-06-02",
    start="2026-01-01 00:00", stop="2026-01-01 01:00",
    instruments=[
        "BINANCE.UM:SWAP:BTCUSDT",
        # "KRAKEN.F:SWAP:BTCUSD",
        # "HYPERLIQUID:SWAP:BTCUSDC",
    ], 
    debug="DEBUG"
);

2026-02-13 15:16:33.326 [ 🐞 ] (runner) [Simulator] :: Preparing simulated trading on ['BINANCE.UM'] for 1000 USDT
2026-02-13 15:16:33.327 [ ℹ️ ] (data) SimulatedDataProvider.BINANCE.UM is initialized
2026-02-13 15:16:33.328 [ ℹ️ ] (processing) Setting event schedule to 1h -1s
2026-02-13 15:16:33.329 [ 🐞 ] (context) [StrategyContext] :: Auto-assigned SimulationTransferManager
2026-02-13 15:16:33.330 [ 🐞 ] (runner) [SimulationRunner] :: Setting warmups: {'ohlc(1h)': '2h'}
2026-01-01 00:00:00.000 [🐞] (runner) [SimulationRunner] :: Running simulation from 2026-01-01 00:00:00 to 2026-01-01 01:00:00
2026-01-01 00:00:00.000 [🐞] (data)  | Warming up ('ohlc(1h)', BINANCE.UM:SWAP:BTCUSDT) -> 2h


Simulating:   0%|          | 0/100 [00:00<?, ?%/s]

2026-01-01 00:00:00.000 [ℹ️] (runner) SimulationRunner ::: Simulation started at 2026-01-01 00:00:00 :::
2026-01-01 00:00:05.000 [🐞] (processing) [ProcessingManager] :: Invoking Test1 on_warmup_finished
2026-01-01 00:00:05.000 [🐞] (processing) [ProcessingManager] :: Test1 warmup finished completed
2026-01-01 00:00:05.000 [🐞] (processing) [ProcessingManager] :: Invoking Test1 on_fit
2026-01-01 00:00:05.000 [🐞] (processing) [ProcessingManager] :: Test1 is fitted


2026-01-01 00:59:59.000 [ℹ️] (1223124595)  === Time: 2026-01-01T00:59:59.000000000 ===
                        open     high      low    close       volume
timestamp                                                           
2025-12-31 14:00:00  88954.3  89192.8  88233.5  88443.4  11096.34400
2025-12-31 15:00:00  88443.4  88450.0  87729.2  87909.0  13304.03200
2025-12-31 16:00:00  87909.0  88024.3  87350.5  87625.9  12077.88600
2025-12-31 17:00:00  87625.9  87716.0  87454.0  87587.2   4255.61430
2025-12-31 18:00:00  87587.2  87952.0  87535.0  87655.8   5108.15330
2025-12-31 19:00:00  87655.9  87846.0  87454.0  87515.0   2612.31900
2025-12-31 20:00:00  87515.0  87683.6  87189.2  87629.0   6228.76900
2025-12-31 21:00:00  87629.0  87841.9  87629.0  87764.0   2124.13900
2025-12-31 22:00:00  87764.0  87849.7  87613.4  87695.7   1465.09900
2025-12-31 23:00:00  87695.8  87702.1  87583.6  87608.2    955.66504
2026-01-01 00:00:00  87608.3  87819.0  87600.0  87773.4   1571.04000
 - - - - - - -
2

In [6]:
rr.read("BTCUSDT", "ohlc(1h)", "2025-12-27 21:00", "2026-01-01 00:00").to_pd().tail(11)

,open,high,low,close,volume,quote_volume,count,taker_buy_volume,taker_buy_quote_volume
timestamp,,,,,,,,,
2025-12-31 14:00:00,88954.3,89192.8,88233.5,88443.4,11096.34400,9.839087e+08,239672,5102.01950,452394208.0
2025-12-31 15:00:00,88443.4,88450.0,87729.2,87909.0,13304.03200,1.171133e+09,257715,6734.04350,592823170.0
2025-12-31 16:00:00,87909.0,88024.3,87350.5,87625.9,12077.88600,1.059027e+09,258188,6296.22750,552186050.0
2025-12-31 17:00:00,87625.9,87716.0,87454.0,87587.2,4255.61430,3.727016e+08,107025,2452.96730,214827680.0
2025-12-31 18:00:00,87587.2,87952.0,87535.0,87655.8,5108.15330,4.485549e+08,101410,3034.10160,266427088.0
2025-12-31 19:00:00,87655.9,87846.0,87454.0,87515.0,2612.31900,2.289705e+08,68740,1116.75590,97909056.0
2025-12-31 20:00:00,87515.0,87683.6,87189.2,87629.0,6228.76900,5.445149e+08,122060,2918.15260,255171232.0
2025-12-31 21:00:00,87629.0,87841.9,87629.0,87764.0,2124.13900,1.863542e+08,44118,1120.77890,98323456.0
2025-12-31 22:00:00,87764.0,87849.7,87613.4,87695.7,1465.09900,1.285583e+08,31504,687.82700,60353484.0


## Update csv storage for test

In [28]:
r1 = StorageRegistry.get("qdb::quantlab")["BINANCE.UM", "SWAP"]
r2 = StorageRegistry.get("qdb::quantlab")["KRAKEN", "SWAP"]
r3 = StorageRegistry.get("qdb::quantlab")["HYPERLIQUID", "SWAP"]

In [ ]:
r1.get_time_range("BTCUSDT", "quote")

In [ ]:
r2.get_time_range("BTCUSD", "quote")

In [ ]:
r3.get_time_range("BTCUSDC", "quote")

In [ ]:
if 0:
    q1 = r1.read("BTCUSDT", "quote", "2026-01-01", "2026-01-03")
    q2 = r2.read("BTCUSD", "quote", "2026-01-01", "2026-01-03")
    q3 = r3.read("BTCUSDC", "quote", "2026-01-01", "2026-01-03")

    q1.to_pd().drop(columns=["symbol"]).to_csv("../../tests/data/storages/multi/BINANCE.UM/SWAP/BTCUSDT.quote.csv.gz")
    q2.to_pd().drop(columns=["symbol"]).to_csv("../../tests/data/storages/multi/KRAKEN.F/SWAP/BTCUSD.quote.csv.gz")
    q3.to_pd().drop(columns=["symbol"]).to_csv("../../tests/data/storages/multi/HYPERLIQUID/SWAP/BTCUSDC.quote.csv.gz")

In [ ]:
if 0:
    sm = "BCH"
    q1 = r1.read(f"{sm}USDT", "ohlc(1h)", "2025-01-01", "2026-01-03")
    q2 = r2.read(f"{sm}USD", "ohlc(1h)", "2025-01-01", "2026-01-03")
    q3 = r3.read(f"{sm}USDC", "ohlc(1h)", "2025-01-01", "2026-01-03")

    q1.to_pd().drop(columns=["symbol"]).to_csv(f"../../tests/data/storages/multi/BINANCE.UM/SWAP/{sm}USDT.1H.csv.gz")
    q2.to_pd().drop(columns=["symbol"]).to_csv(f"../../tests/data/storages/multi/KRAKEN.F/SWAP/{sm}USD.1H.csv.gz")
    q3.to_pd().drop(columns=["symbol"]).to_csv(f"../../tests/data/storages/multi/HYPERLIQUID/SWAP/{sm}USDC.1H.csv.gz")